# 🔴 Solution: Linear Self-Attention（参考版）

## 📋 题目描述

**难度：** Hard

实现基于核特征映射的线性自注意力。

线性注意力用特征映射 phi 替代 softmax，通过重排计算顺序将复杂度从 O(S^2*D) 降至 O(S*D^2)。

**签名:** `linear_attention(Q, K, V) -> Tensor`

**参数:**
- `Q` — 查询张量 (B, S, D_k)
- `K` — 键张量 (B, S, D_k)
- `V` — 值张量 (B, S, D_v)

**返回:** 注意力输出 (B, S, D_v)

**约束:**
- 特征映射：`phi(x) = elu(x) + 1`
- 计算 `phi(Q) @ (phi(K)^T @ V)` 而非 `softmax(Q @ K^T) @ V`
- 通过 `phi(Q) @ sum(phi(K))` 归一化（加 `eps=1e-6` 保证数值稳定）

**提示：** `phi(x) = elu(x) + 1`。输出 = `(phi(Q) @ (phi(K).T @ V)) / (phi(Q) @ phi(K).sum(dim=-2, keepdim=True).T)`


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def linear_attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
    # Linear Attention
    # 标准注意力：softmax(QK^T)V，需要计算 S×S 矩阵 → O(S²D)
    # 线性注意力：用核函数近似 softmax，改变计算顺序 → O(SD²)
    # 关键技巧：φ(Q)(φ(K)^T V) 先算 KV 再乘 Q

    d_k = key.size(-1)

    # ---- Step 1: 核函数映射 ----
    # 用 elu + 1 近似 softmax 的核函数
    # elu(x) + 1 保证非负（类似 exp 的效果但计算更快）
    # φ(x) = elu(x) + 1
    Q = torch.nn.functional.elu(query) + 1
    K = torch.nn.functional.elu(key) + 1

    # ---- Step 2: 计算 KV 矩阵（核心优化） ----
    # 标准：Q @ K^T @ V → O(S²D)
    # 线性：K^T @ V → O(SD²)，然后 Q @ (K^T V) → O(SD²)
    # K^T: [B, H, D, S] @ V: [B, H, S, D] → KV: [B, H, D, D]
    # 这是 O(SD²) 而非 O(S²D)，当 S > D 时更快
    KV = K.transpose(-2, -1) @ value

    # ---- Step 3: Q 乘以 KV ----
    # Q: [B, H, S, D] @ KV: [B, H, D, D] → output: [B, H, S, D]
    output = Q @ KV

    # ---- Step 4: 归一化 ----
    # 除以 Z = φ(K) 的行和，起到类似 softmax 分母的作用
    # Z: [B, H, S, 1]
    Z = Q @ K.transpose(-2, -1).sum(dim=-1, keepdim=True)
    output = output / (Z + 1e-6)

    return output

In [ ]:
Q=torch.randn(1,8,16); K=torch.randn(1,8,16); V=torch.randn(1,8,32)
print('Shape:', linear_attention(Q,K,V).shape)

## 📝 核心思路总结

1. **核心思想**：改变矩阵乘法顺序，O(S²D) → O(SD²)
2. **核函数**：elu+1 近似 exp，保持非负性
3. **结合律**：Q(K^TV) 先算 KV 再乘 Q
4. **归一化**：除以 Z 保证数值合理性